# Tutorial: Exploring TCGA Data with the GDC Client

This notebook walks through how to query TCGA (The Cancer Genome Atlas) data using our GDC client.

**What you'll learn:**
- What data is available in TCGA
- How to discover fields dynamically (no memorizing!)
- How to query patients and their clinical data
- How to find slide images and other files

## 1. Setup

First, let's import the client and create a connection. No authentication needed for open-access data.

In [11]:
import sys
sys.path.insert(0, '../../../..')  # Add project root to path

from src.data.tcga import GDCClient

client = GDCClient()
print("Connected to GDC API")

Connected to GDC API


## 2. What Cancer Types Are Available?

TCGA has 33 cancer types. Let's see them all.

In [12]:
projects = client.list_projects(program="TCGA")

print(f"Found {len(projects)} TCGA projects\n")
print(f"{'Project':<12} | {'Patients':>8} | {'Files':>8} | Cancer Type")
print("-" * 70)

for p in projects:
    disease = p.disease_type[0] if p.disease_type else "N/A"
    print(f"{p.project_id:<12} | {p.case_count:>8} | {p.file_count:>8} | {disease[:35]}")

Found 33 TCGA projects

Project      | Patients |    Files | Cancer Type
----------------------------------------------------------------------
TCGA-LGG     |      516 |    33453 | Gliomas
TCGA-BRCA    |     1098 |    70774 | Adnexal and Skin Appendage Neoplasm
TCGA-LAML    |      200 |     8839 | Myeloid Leukemias
TCGA-UCS     |       57 |     3720 | Basal Cell Neoplasms
TCGA-GBM     |      617 |    30326 | Not Reported
TCGA-THYM    |      124 |     7968 | Thymic Epithelial Neoplasms
TCGA-TGCT    |      263 |    12851 | Germ Cell Neoplasms
TCGA-PCPG    |      179 |    11823 | Paragangliomas and Glomus Tumors
TCGA-CHOL    |       51 |     3171 | Adenomas and Adenocarcinomas
TCGA-DLBC    |       58 |     3141 | Mature B-Cell Lymphomas
TCGA-CESC    |      307 |    19315 | Squamous Cell Neoplasms
TCGA-ESCA    |      185 |    11120 | Squamous Cell Neoplasms
TCGA-ACC     |       92 |     5789 | Adenomas and Adenocarcinomas
TCGA-KICH    |      113 |     5993 | Adenomas and Adenocarcinomas
TC

## 3. Discovering Available Fields

You don't need to memorize field names. The client can tell you what's available.

**Key concept:** Fields can be "expanded" to include nested data (like patient demographics, diagnoses, samples).

In [4]:
# What nested data can we expand for cases (patients)?
expandable = client.get_expandable_fields("cases")

print("Expandable fields for CASES (patients):")
print("These are nested objects you can include in queries\n")

for field in expandable:
    print(f"  - {field}")

Expandable fields for CASES (patients):
These are nested objects you can include in queries

  - annotations
  - demographic
  - diagnoses
  - diagnoses.annotations
  - diagnoses.pathology_details
  - diagnoses.treatments
  - exposures
  - family_histories
  - files
  - files.analysis
  - files.analysis.input_files
  - files.analysis.metadata
  - files.analysis.metadata.read_groups
  - files.analysis.metadata.read_groups.read_group_qcs
  - files.archive
  - files.center
  - files.downstream_analyses
  - files.downstream_analyses.output_files
  - files.index_files
  - files.metadata_files
  - follow_ups
  - follow_ups.molecular_tests
  - follow_ups.other_clinical_attributes
  - project
  - project.program
  - samples
  - samples.annotations
  - samples.portions
  - samples.portions.analytes
  - samples.portions.analytes.aliquots
  - samples.portions.analytes.aliquots.annotations
  - samples.portions.analytes.aliquots.center
  - samples.portions.analytes.annotations
  - samples.portions.

In [28]:
# What are ALL available fields? (there are hundreds)
all_fields = client.discover_fields("cases")

print(f"Total available fields: {len(all_fields)}\n")
print("First 30 fields:")
for f in all_fields:
    print(f"  {f}")
print(f"\n... and {len(all_fields) - 30} more")

Total available fields: 1172

First 30 fields:
  aliquot_ids
  analyte_ids
  annotations.annotation_id
  annotations.case_id
  annotations.case_submitter_id
  annotations.category
  annotations.classification
  annotations.created_datetime
  annotations.creator
  annotations.entity_id
  annotations.entity_submitter_id
  annotations.entity_type
  annotations.legacy_created_datetime
  annotations.legacy_updated_datetime
  annotations.notes
  annotations.state
  annotations.status
  annotations.submitter_id
  annotations.updated_datetime
  case_autocomplete
  case_id
  consent_type
  created_datetime
  days_to_consent
  days_to_lost_to_followup
  demographic.age_at_index
  demographic.age_is_obfuscated
  demographic.cause_of_death
  demographic.cause_of_death_source
  demographic.country_of_birth
  demographic.country_of_residence_at_enrollment
  demographic.created_datetime
  demographic.days_to_birth
  demographic.days_to_death
  demographic.demographic_id
  demographic.education_level


In [29]:
from src.data.tcga import GDCClient                                                                                                                                                                     
import pandas as pd                                                                                                                                                                                     
                                                                                                                                                                                                        
client = GDCClient()                                                                                                                                                                                  

projects = ["TCGA-LUAD", "TCGA-LUSC"]
all_slides = []

for project_id in projects:
    print(f"Fetching {project_id}...")

    # Use our client's _paginate with expand
    hits = client._paginate(
        "files",
        filters={
            "op": "and",
            "content": [
                {"op": "=", "content": {"field": "cases.project.project_id", "value": project_id}},
                {"op": "=", "content": {"field": "data_type", "value": "Slide Image"}},
                {"op": "=", "content": {"field": "access", "value": "open"}}
            ]
        },
        expand=["cases", "cases.samples", "cases.demographic", "cases.diagnoses", "associated_entities"]
    )

    print(f"  Found {len(hits)} slides")

    for hit in hits:
        case = hit.get("cases", [{}])[0]
        demo = case.get("demographic", {}) or {}
        diag = (case.get("diagnoses", []) or [{}])[0]
        samples = case.get("samples", [])
        assoc = (hit.get("associated_entities", []) or [{}])[0]

        slide_submitter = assoc.get("entity_submitter_id", "")
        matched_sample = {}
        for s in samples:
            if slide_submitter.startswith(s.get("submitter_id", "XXX")):
                matched_sample = s
                break

        all_slides.append({
            "file_id": hit.get("file_id"),
            "filename": hit.get("file_name"),
            "file_size": hit.get("file_size"),
            "slide_id": assoc.get("entity_id"),
            "slide_submitter_id": slide_submitter,
            "sample_id": matched_sample.get("sample_id"),
            "sample_submitter_id": matched_sample.get("submitter_id"),
            "sample_type": matched_sample.get("sample_type"),
            "tissue_type": matched_sample.get("tissue_type"),
            "case_id": case.get("case_id"),
            "case_submitter_id": case.get("submitter_id"),
            "project_id": project_id,
            "gender": demo.get("gender"),
            "race": demo.get("race"),
            "ethnicity": demo.get("ethnicity"),
            "year_of_birth": demo.get("year_of_birth"),
            "primary_diagnosis": diag.get("primary_diagnosis"),
            "tumor_stage": diag.get("tumor_stage"),
            "tumor_grade": diag.get("tumor_grade"),
            "age_at_diagnosis": diag.get("age_at_diagnosis"),
            "vital_status": diag.get("vital_status"),
            "days_to_death": diag.get("days_to_death"),
        })

df = pd.DataFrame(all_slides)
print(df)


Fetching TCGA-LUAD...
  Found 1608 slides
Fetching TCGA-LUSC...
  Found 1612 slides
Fetching TCGA-LGG...
  Found 1572 slides
Fetching TCGA-GBM...
  Found 2053 slides
                                   file_id  \
0     6a0ea716-a5f2-47f3-880b-537a5cdc2324   
1     b059ce82-63d7-43b4-b52b-a681daeaef5a   
2     c5cc9280-d4c9-4090-9fb1-1d1cc5e2f8d6   
3     22a6cf23-604a-4163-aa84-415fa93d6f58   
4     1ec9435b-6056-4d34-802d-a4d20208f60e   
...                                    ...   
6840  41031406-a192-4896-a5b7-2ad4edf2d588   
6841  0fb8169d-af6d-4a9a-91ec-25c9485633cc   
6842  9e3dbd43-cda8-4893-8a2f-d850b83d88b1   
6843  dedab559-28df-466c-9ad4-72f5648eeacf   
6844  2ac2daa7-a111-4013-9f17-1371be4655c0   

                                               filename   file_size  \
0     TCGA-86-8074-01Z-00-DX1.0c34b434-8701-4060-a4e...   532458405   
1     TCGA-86-8074-01A-01-BS1.9b6a32f7-07bb-4e59-aa6...  1667746421   
2     TCGA-78-7535-01Z-00-DX1.c4ca06f3-22d1-4e39-85c...  1281854239 

In [52]:
from src.data.tcga import GDCClient
import pandas as pd

client = GDCClient()

projects = ["TCGA-LUAD", "TCGA-LUSC"]
all_slides = []

for project_id in projects:
    print(f"Fetching {project_id}...")

    hits = client._paginate(
        "files",
        filters={
            "op": "and",
            "content": [
                {"op": "=", "content": {"field": "cases.project.project_id", "value":
project_id}},
                {"op": "=", "content": {"field": "data_type", "value": "Slide Image"}},
                {"op": "=", "content": {"field": "access", "value": "open"}}
            ]
        },
        expand=["cases", "cases.samples", "cases.demographic", "cases.diagnoses",
"associated_entities"],
          # limit for testing
    )

    print(f"  Found {len(hits)} slides")

    for hit in hits:
        case = hit.get("cases", [{}])[0]
        demo = case.get("demographic", {}) or {}
        diagnoses = case.get("diagnoses", []) or []
        samples = case.get("samples", [])
        assoc = (hit.get("associated_entities", []) or [{}])[0]

        # Find PRIMARY diagnosis
        primary_diag = {}
        for d in diagnoses:
            if d.get("diagnosis_is_primary_disease") in ["Yes", True, "yes", "TRUE"]:
                primary_diag = d
                break
        if not primary_diag and diagnoses:
            primary_diag = diagnoses[0]

        # Match sample
        slide_submitter = assoc.get("entity_submitter_id", "")
        matched_sample = {}
        for s in samples:
            if slide_submitter.startswith(s.get("submitter_id", "XXX")):
                matched_sample = s
                break

        all_slides.append({
            "file_id": hit.get("file_id"),
            "filename": hit.get("file_name"),
            "slide_id": assoc.get("entity_id"),
            "slide_submitter_id": slide_submitter,
            "sample_id": matched_sample.get("sample_id"),
            "sample_type": matched_sample.get("sample_type"),
            "tissue_type": matched_sample.get("tissue_type"),
            "case_id": case.get("case_id"),
            "case_submitter_id": case.get("submitter_id"),
            "project_id": project_id,
            "gender": demo.get("gender"),
            "race": demo.get("race"),
            "primary_diagnosis": primary_diag.get("primary_diagnosis"),
            "diagnosis_is_primary": primary_diag.get("diagnosis_is_primary_disease"),
            "tumor_stage": primary_diag.get("tumor_stage"),
            "vital_status": primary_diag.get("vital_status"),
        })

df = pd.DataFrame(all_slides)
print(df[["project_id", "primary_diagnosis",
"diagnosis_is_primary"]].drop_duplicates())

Fetching TCGA-LUAD...
  Found 1608 slides
Fetching TCGA-LUSC...
  Found 1612 slides
     project_id                                  primary_diagnosis  \
0     TCGA-LUAD                      Papillary adenocarcinoma, NOS   
2     TCGA-LUAD                                Adenocarcinoma, NOS   
6     TCGA-LUAD                 Adenocarcinoma with mixed subtypes   
18    TCGA-LUAD        Bronchiolo-alveolar carcinoma, non-mucinous   
22    TCGA-LUAD            Bronchiolo-alveolar adenocarcinoma, NOS   
26    TCGA-LUAD                            Mucinous adenocarcinoma   
49    TCGA-LUAD                               Solid carcinoma, NOS   
80    TCGA-LUAD                              Acinar cell carcinoma   
87    TCGA-LUAD                         Signet ring cell carcinoma   
112   TCGA-LUAD                     Clear cell adenocarcinoma, NOS   
144   TCGA-LUAD                  Invasive micropapillary carcinoma   
309   TCGA-LUAD              Bronchio-alveolar carcinoma, mucinous   
1608  

primary_diagnosis
Squamous cell carcinoma, NOS                                 1515
Adenocarcinoma, NOS                                          1028
Adenocarcinoma with mixed subtypes                            298
Papillary adenocarcinoma, NOS                                  71
Mucinous adenocarcinoma                                        54
Bronchiolo-alveolar carcinoma, non-mucinous                    50
Acinar cell carcinoma                                          50
Basaloid squamous cell carcinoma                               43
Squamous cell carcinoma, keratinizing, NOS                     29
Bronchio-alveolar carcinoma, mucinous                          15
Clear cell adenocarcinoma, NOS                                 11
Solid carcinoma, NOS                                           11
Invasive micropapillary carcinoma                              10
Bronchiolo-alveolar adenocarcinoma, NOS                        10
Papillary squamous cell carcinoma                         

In [69]:
hits = client._paginate(
    "files",
    filters={
        "op": "and",
        "content": [
            {"op": "=", "content": {"field": "cases.project.project_id", "value":
"TCGA-LUAD"}},
            {"op": "=", "content": {"field": "data_type", "value": "Slides"}}
        ]
    },  # The actual link name from data dictionary
    expand=["associated_entities"],
    max_results=1
)

print(hits[0])




IndexError: list index out of range

## 4. Getting Patient Data

Let's get some actual patients from TCGA-BRCA (Breast Cancer).

We use `expand=` to tell the API what nested data to include.

In [ ]:
# Get 5 breast cancer patients with their clinical data
cases = client.get_cases(
    project_id="TCGA-BRCA",
    expand=["demographic", "diagnoses", "samples"],
    max_results=5
)

print(f"Retrieved {len(cases)} patients\n")

for c in cases:
    print(f"Patient: {c.submitter_id}")
    print(f"  Gender: {c.gender}")
    print(f"  Race: {c.race}")
    print(f"  Age at diagnosis: {c.age_at_diagnosis} days (~{c.age_at_diagnosis//365 if c.age_at_diagnosis else '?'} years)")
    print(f"  Cancer: {c.primary_diagnosis}")
    print(f"  Stage: {c.tumor_stage}")
    print(f"  Status: {c.vital_status}")
    print(f"  Samples collected: {len(c.samples)}")
    print(f"  Slides available: {len(c.slide_ids)}")
    print()

### Looking at Raw Data

Each case object has a `_raw` attribute with the complete API response. Useful for exploring what's really there.

In [ ]:
# Look at raw data for first patient
import json

case = cases[0]
print(f"Raw data keys for {case.submitter_id}:")
print(list(case._raw.keys()))

print("\nDemographic data:")
print(json.dumps(case._raw.get('demographic', {}), indent=2))

print("\nFirst diagnosis:")
if case._raw.get('diagnoses'):
    print(json.dumps(case._raw['diagnoses'][0], indent=2))

## 5. Filtering Patients

You can filter by various criteria like gender, vital status, etc.

In [ ]:
# Find deceased female patients
deceased = client.get_cases(
    project_id="TCGA-BRCA",
    gender="female",
    vital_status="Dead",
    expand=["demographic", "diagnoses"],
    max_results=10
)

print(f"Found {len(deceased)} deceased female patients\n")

for c in deceased:
    days = c.days_to_death or "unknown"
    years = f"({c.days_to_death // 365} years)" if c.days_to_death else ""
    print(f"{c.submitter_id}: died after {days} days {years}")

## 6. Finding Files (Slides, Clinical Data, etc.)

Let's see what file types are available and get some slide images.

In [ ]:
# What data types exist in TCGA-BRCA?
data_types = client.get_available_data_types(project_id="TCGA-BRCA")

print("Available data types in TCGA-BRCA:\n")
for dt in data_types:
    print(f"  - {dt}")

In [ ]:
# Get some slide images (open access only)
slides = client.get_slide_images(
    project_id="TCGA-BRCA",
    access="open",
    max_results=5
)

print(f"Found {len(slides)} slide images\n")

for s in slides:
    size_gb = s.file_size / 1e9
    print(f"File: {s.filename}")
    print(f"  Patient: {s.case_submitter_id}")
    print(f"  Type: {s.experimental_strategy}")
    print(f"  Size: {size_gb:.2f} GB")
    print(f"  Format: {s.data_format}")
    print(f"  Download URL: {client.get_download_url(s.file_id)}")
    print()

In [ ]:
# Get clinical supplement files (XMLs with detailed clinical info)
clinical_files = client.get_clinical_files(
    project_id="TCGA-BRCA",
    access="open",
    max_results=5
)

print(f"Found {len(clinical_files)} clinical files\n")

for f in clinical_files:
    print(f"{f.filename}")
    print(f"  Type: {f.data_type}")
    print(f"  Format: {f.data_format}")
    print()

In [9]:
from src.data.tcga import GDCClient
client = GDCClient()

  # Query with expand to get nested case/sample data
files = client.get_files(
      project_id="TCGA-BRCA",
      data_type="Slide Image",
      access="open",
      max_results=1,
      fields=None  # Get default fields
  )

  # Look at the raw response
import json
print(json.dumps(files[0]._raw, indent=2))

{
  "id": "495ab2ae-0286-4d87-8c7b-4d4af7eded01",
  "cases": [
    {
      "case_id": "878f975b-94fd-4d69-b7e7-1ed3ac2ee438",
      "submitter_id": "TCGA-BH-A18H",
      "project": {
        "project_id": "TCGA-BRCA"
      }
    }
  ],
  "access": "open",
  "file_name": "TCGA-BH-A18H-01A-01-TSA.75dba5a3-f9f5-4ff4-814c-7d2451820e03.svs",
  "file_id": "495ab2ae-0286-4d87-8c7b-4d4af7eded01",
  "data_type": "Slide Image",
  "data_category": "Biospecimen",
  "file_size": 217303029
}


In [10]:
response = client._request("files", {                                                                                        
    "filters": json.dumps({
        "op": "and",
        "content": [
            {"op": "=", "content": {"field": "cases.project.project_id", "value": "TCGA-BRCA"}},
            {"op": "=", "content": {"field": "data_type", "value": "Slide Image"}},
            {"op": "=", "content": {"field": "access", "value": "open"}}
        ]
    }),
    "expand": "cases,cases.samples,associated_entities",
    "size": 1
})
print(json.dumps(response["data"]["hits"][0], indent=2))

{
  "id": "495ab2ae-0286-4d87-8c7b-4d4af7eded01",
  "data_format": "SVS",
  "cases": [
    {
      "lost_to_followup": "No",
      "primary_site": "Breast",
      "disease_type": "Ductal and Lobular Neoplasms",
      "updated_datetime": "2025-01-05T22:44:58.621704-06:00",
      "case_id": "878f975b-94fd-4d69-b7e7-1ed3ac2ee438",
      "submitter_id": "TCGA-BH-A18H",
      "index_date": "Diagnosis",
      "state": "released",
      "days_to_consent": -32,
      "created_datetime": null,
      "samples": [
        {
          "intermediate_dimension": null,
          "tumor_descriptor": "Primary",
          "time_between_clamping_and_freezing": null,
          "sample_id": "9321161b-d568-4d38-aed3-956786a06329",
          "freezing_method": null,
          "pathology_report_uuid": "DF536409-601F-489B-9F8A-A2B372BF9999",
          "submitter_id": "TCGA-BH-A18H-01A",
          "tumor_code_id": null,
          "shortest_dimension": null,
          "sample_type": "Primary Tumor",
          "c

## 7. Creating a Download Manifest

To download files in bulk, you create a manifest and use the `gdc-client` tool.

In [ ]:
# Get slides and create a manifest
slides = client.get_slide_images("TCGA-BRCA", access="open", max_results=3)

# Create manifest (just display it, don't save)
manifest = client.create_manifest(slides)

print("Download manifest:")
print("-" * 80)
print(manifest)
print("-" * 80)

print("\nTo download these files:")
print("1. Save manifest to file: manifest.txt")
print("2. Run: gdc-client download -m manifest.txt")

## 8. Summary

**Key methods:**

| Method | What it does |
|--------|-------------|
| `list_projects()` | List all cancer types |
| `discover_fields(endpoint)` | See all available fields |
| `get_expandable_fields(endpoint)` | See what nested data is available |
| `get_cases(project_id, expand=[...])` | Get patient clinical data |
| `get_slide_images(project_id)` | Get pathology slides |
| `get_clinical_files(project_id)` | Get clinical XMLs |
| `get_available_data_types(project_id)` | See what file types exist |
| `create_manifest(files)` | Create download manifest |

**Next steps:**
- Run the test suite: `python src/data/tcga/gdc_client.py`
- See the README for more details: `src/data/tcga/README.md`